In [1]:
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from app.graphs.simple_agent import graph

### RAGAS Set Up

In [2]:
from __future__ import annotations

import asyncio
import json
import os
from concurrent.futures import ThreadPoolExecutor
from getpass import getpass
from pathlib import Path
from typing import Annotated, Any
from uuid import uuid4

import instructor
import pandas as pd
import requests
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document
from langchain_core.messages import (
    AIMessage as LCAIMessage,
    HumanMessage as LCHumanMessage,
    SystemMessage as LCSystemMessage,
    ToolMessage as LCToolMessage,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from openai import OpenAI
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.messages import (
    AIMessage as RagasAIMessage,
    HumanMessage as RagasHumanMessage,
    ToolCall as RagasToolCall,
    ToolMessage as RagasToolMessage,
)
from ragas.metrics.collections import (
    AgentGoalAccuracyWithReference,
    AnswerAccuracy,
    ContextEntityRecall,
    ContextRecall,
    Faithfulness,
    NoiseSensitivity,
    ToolCallAccuracy,
    TopicAdherence,
)
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.transforms import (
    CustomNodeFilter,
    default_transforms_for_prechunked,
)
from typing_extensions import TypedDict

In [3]:
load_dotenv()

SESSION6_RUNTIME_REVISION = "ragas-sync-adapter-v4"


def read_required_secret(names: tuple[str, ...], prompt: str) -> str:
    for name in names:
        if value := os.environ.get(name):
            return value

    value = getpass(prompt)
    os.environ[names[0]] = value
    return value


gateway_api_key = read_required_secret(
    ("AI_GATEWAY_API_KEY", "VERCEL_OIDC_TOKEN"),
    "Vercel AI Gateway API key: ",
)

GATEWAY_BASE_URL = os.environ.get(
    "AIM_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)

GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)

JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)

EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)

gateway_sync_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
# Ragas generation needs structured output for graph transforms and sample creation.
generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_sync_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=2048,
)
generator_llm.model_args = {"max_tokens": 2048, "max_retries": 3}
generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_sync_client,
)

TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "4"))
EVAL_CASE_LIMIT = int(os.environ.get("AIM_RAG_EVAL_LIMIT", "3"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

In [4]:
def build_sync_judge_llm():
    # Construct inside each scoring worker so a notebook cannot reuse a
    # previous kernel's async client by accident.
    judge = llm_factory(
        JUDGE_MODEL_NAME,
        provider="openai",
        client=OpenAI(api_key=gateway_api_key, base_url=GATEWAY_BASE_URL),
        mode=instructor.Mode.TOOLS,
        max_tokens=4096,
    )
    judge.model_args = {"max_tokens": 4096, "max_retries": 3}

    async def agenerate_from_sync(prompt, response_model):
        return await asyncio.to_thread(
            judge.generate,
            prompt=prompt,
            response_model=response_model,
        )

    judge.agenerate = agenerate_from_sync
    return judge


judge_llm = build_sync_judge_llm()
if judge_llm.is_async:
    raise RuntimeError("Session 6 requires a synchronous Ragas judge client.")

# Repair a previously executed Task 6 cell when this configuration cell is
# rerun in an existing notebook kernel.
if "rag_metrics" in globals():
    rag_metrics = {
        "context_recall": ContextRecall(llm=judge_llm),
        "faithfulness": Faithfulness(llm=judge_llm),
        "answer_accuracy": AnswerAccuracy(llm=judge_llm),
        "context_entity_recall": ContextEntityRecall(llm=judge_llm),
        "noise_sensitivity": NoiseSensitivity(llm=judge_llm, mode="relevant"),
    }


ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

# Jupyter owns an event loop. Run Ragas coroutines in a dedicated worker;
# their model requests use the synchronous adapter above.
def run_ragas_sync(func, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(func, *args, **kwargs).result()


def run_ragas_async(async_function, *args, **kwargs):
    # Accept both the current function-plus-arguments form and the older
    # pre-v4 coroutine form so a partially rerun notebook can recover.
    if asyncio.iscoroutine(async_function):
        return run_ragas_sync(asyncio.run, async_function)

    def invoke():
        return asyncio.run(async_function(*args, **kwargs))

    return run_ragas_sync(invoke)

In [6]:
corpus_path = Path("data/HealthWellnessGuide.txt")
if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

source_documents = TextLoader(str(corpus_path), encoding="utf-8").load()
generation_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900,
    chunk_overlap=100,
)
generation_chunks = generation_splitter.split_documents(source_documents)

print(f"Source documents: {len(source_documents)}")
print(f"Generation chunks: {len(generation_chunks)}")
print(generation_chunks[0].page_content[:900])

Source documents: 1
Generation chunks: 7
# Health & Wellness Guide: Course Evaluation Corpus

This short course corpus is for learning retrieval and evaluation. It offers
general wellness information, not diagnosis, treatment, or individualized
medical, nutrition, or mental-health advice. A reader with persistent,
concerning, or worsening symptoms should consult a qualified health
professional. Seek urgent help for emergencies.

## Movement: build a routine gradually

For many adults, the public-health target is at least 150 minutes of
moderate-intensity aerobic activity each week, or 75 minutes of
vigorous-intensity activity, plus muscle-strengthening activity on two or more
days each week. The time can be spread across the week and broken into smaller
sessions. Some activity is generally better than none, and a gradual increase
can make a new routine more manageable.


In [7]:
generation_transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
# Ragas' generation transforms make internal async Instructor calls. Keep
# them off the Jupyter event loop for the same reason as the metric calls.
synthetic_testset = run_ragas_sync(
    testset_generator.generate_with_chunks,
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=generation_transforms,
    run_config=ragas_run_config,
)
synthetic_testset_df = synthetic_testset.to_pandas()

synthetic_testset_df[[
    "user_input",
    "reference",
    "reference_contexts",
]].head()

Applying SummaryExtractor:   0%|          | 0/7 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/21 [00:00<?, ?it/s]

c:\Personal Files\Data Science\homework_local\10_LLM_Servers\.venv\Lib\site-packages\ragas\testset\transforms\base.py:198: UserWarning: Using sync embedding model OpenAIEmbeddings in async context. This may impact performance. Consider using an async-compatible embedding model for better performance.
  property_name, property_value = await self.extract(node)


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Skipping multi_hop_abstract_query_synthesizer due to unexpected error: No relationships match the provided condition. Cannot form clusters.


Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/4 [00:00<?, ?it/s]

,user_input,reference,reference_contexts
0,"What is the Course Evaluation Corpus for, and ...",The Course Evaluation Corpus is for learning r...,[# Health & Wellness Guide: Course Evaluation ...
1,"Can brisk walking be a moderate activity, and ...",Yes. Brisk walking can be an example of modera...,[Examples of moderate activity can include bri...
2,What is the Sleep section about?,The provided context includes only the heading...,[## Movement: daily practicality\n\nChoose mov...
3,What does the CDC say about sleep needs and pr...,CDC guidance says adults ages 18 to 60 general...,[## Sleep: routine and environment\n\nSleep su...


In [8]:
required_testset_columns = [
    "user_input",
    "reference",
    "reference_contexts",
]
missing_columns = set(required_testset_columns) - set(synthetic_testset_df.columns)
if missing_columns:
    raise RuntimeError(
        "Ragas did not return the expected evaluation columns: "
        f"{sorted(missing_columns)}"
    )

# In a production workflow, replace this with an explicit reviewed-status filter.
reviewed_testset_df = (
    synthetic_testset_df.loc[:, required_testset_columns]
    .head(EVAL_CASE_LIMIT)
    .copy()
)
reviewed_testset_df

,user_input,reference,reference_contexts
0,"What is the Course Evaluation Corpus for, and ...",The Course Evaluation Corpus is for learning r...,[# Health & Wellness Guide: Course Evaluation ...
1,"Can brisk walking be a moderate activity, and ...",Yes. Brisk walking can be an example of modera...,[Examples of moderate activity can include bri...
2,What is the Sleep section about?,The provided context includes only the heading...,[## Movement: daily practicality\n\nChoose mov...


### Elaluate RAG with RAGAS

In [13]:
def as_context_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        return [value]
    return [str(item) for item in value]


def message_text(message: Any) -> str:
    content = message.content
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts: list[str] = []
        for block in content:
            if isinstance(block, str):
                parts.append(block)
            elif isinstance(block, dict):
                text = block.get("text") or block.get("content")
                if isinstance(text, str):
                    parts.append(text)
        return "".join(parts)
    return str(content)


def extract_retrieved_contexts(messages: list[Any]) -> list[str]:
    contexts: list[str] = []
    for msg in messages:
        if isinstance(msg, ToolMessage):
            contexts.append(message_text(msg))
    return contexts


def run_rag_over_testset(graph, dataframe: pd.DataFrame) -> list[dict[str, Any]]:
    rows = []
    for _, row in dataframe.iterrows():
        try:
            result = graph.invoke({"messages": [HumanMessage(content=row["user_input"])]})
        except Exception as exc:
            message = str(exc)
            if "Request too large" in message or "413" in message or "rate limit" in message.lower():
                rows.append(
                    {
                        "user_input": row["user_input"],
                        "reference": row["reference"],
                        "reference_contexts": as_context_list(row["reference_contexts"]),
                        "retrieved_contexts": [],
                        "response": "I couldn't answer because the model request was rejected by the provider.",
                    }
                )
                continue
            raise

        retrieved_contexts = extract_retrieved_contexts(result["messages"])
        if not retrieved_contexts:
            rows.append(
                {
                    "user_input": row["user_input"],
                    "reference": row["reference"],
                    "reference_contexts": as_context_list(row["reference_contexts"]),
                    "retrieved_contexts": [],
                    "response": message_text(result["messages"][-1]) if result.get("messages") else "No retrieved context was returned by the graph.",
                }
            )
            continue

        rows.append(
            {
                "user_input": row["user_input"],
                "reference": row["reference"],
                "reference_contexts": as_context_list(row["reference_contexts"]),
                "retrieved_contexts": retrieved_contexts,
                "response": message_text(result["messages"][-1]),
            }
        )
    return rows


baseline_rows = run_rag_over_testset(graph, reviewed_testset_df)
pd.DataFrame(baseline_rows)[["user_input", "response", "reference"]]

,user_input,response,reference
0,"What is the Course Evaluation Corpus for, and ...",I couldn't answer because the model request wa...,The Course Evaluation Corpus is for learning r...
1,"Can brisk walking be a moderate activity, and ...",**Yes – brisk walking is considered a moderate...,Yes. Brisk walking can be an example of modera...
2,What is the Sleep section about?,I’m not sure which “Sleep” section you’re refe...,The provided context includes only the heading...


In [16]:
async def score_rag_rows(rows: list[dict[str, Any]]) -> pd.DataFrame:
    judge_llm = build_sync_judge_llm()
    rag_metrics = {
        "context_recall": ContextRecall(llm=judge_llm),
        "faithfulness": Faithfulness(llm=judge_llm),
        "answer_accuracy": AnswerAccuracy(llm=judge_llm),
        "context_entity_recall": ContextEntityRecall(llm=judge_llm),
        "noise_sensitivity": NoiseSensitivity(llm=judge_llm, mode="relevant"),
    }

    score_rows = []
    for index, row in enumerate(rows, start=1):
        retrieved_contexts = row.get("retrieved_contexts") or []
        if not retrieved_contexts:
            score_rows.append(
                {
                    "case": index,
                    "context_recall": 0.0,
                    "faithfulness": 0.0,
                    "answer_accuracy": 0.0,
                    "context_entity_recall": 0.0,
                    "noise_sensitivity": 0.0,
                }
            )
            continue

        score_rows.append(
            {
                "case": index,
                "context_recall": (await rag_metrics["context_recall"].ascore(
                    user_input=row["user_input"],
                    retrieved_contexts=retrieved_contexts,
                    reference=row["reference"],
                )).value,
                "faithfulness": (await rag_metrics["faithfulness"].ascore(
                    user_input=row["user_input"],
                    response=row["response"],
                    retrieved_contexts=retrieved_contexts,
                )).value,
                "answer_accuracy": (await rag_metrics["answer_accuracy"].ascore(
                    user_input=row["user_input"],
                    response=row["response"],
                    reference=row["reference"],
                )).value,
                "context_entity_recall": (await rag_metrics["context_entity_recall"].ascore(
                    reference=row["reference"],
                    retrieved_contexts=retrieved_contexts,
                )).value,
                "noise_sensitivity": (await rag_metrics["noise_sensitivity"].ascore(
                    user_input=row["user_input"],
                    response=row["response"],
                    reference=row["reference"],
                    retrieved_contexts=retrieved_contexts,
                )).value,
            }
        )
    return pd.DataFrame(score_rows)


baseline_scores = run_ragas_async(score_rag_rows, baseline_rows)
baseline_summary = baseline_scores.drop(columns="case").mean().to_frame("baseline")
baseline_summary

,baseline
context_recall,0.111111
faithfulness,0.100000
answer_accuracy,0.166667
context_entity_recall,0.000000
noise_sensitivity,0.083333
